# 📊 NeMo Fraud Detection: Advanced LLM-as-a-Judge Evaluation with W&B Lineage

This notebook performs an advanced LLM-powered evaluation (**LLM-as-a-Judge**) with structured chain-of-thought analysis and logs all metrics, dataframes, and report artifacts to **Weights & Biases (W&B)**.

### Features & W&B Lineage Integration:
1. **Upstream Artifact Lineage:** Dynamically consumes `curated-sft-dataset:latest` via `run.use_artifact(...)` to evaluate directly on the curated validation/test splits.
2. **Structured Analysis (Worker Model):** Analyzes transcripts based on emotions/pressure, security risks, and result.
3. **Automated Quality Assessor (Judge Model):** Independently verifies whether the analysis matches the expected label.
4. **W&B Observability:** Logs KPIs, interactive tables, versioned CSV artifacts with `job_type="llm-judge-evaluation"`, and displays results via collapsible UI elements.

In [1]:
import json
import re
import requests
import pandas as pd
import os
import wandb
import html
from pathlib import Path
from IPython.display import display, HTML

# Configuration of endpoints and default parameters
NIM_URL = "http://172.17.0.1:8800/v1/chat/completions"
MODEL_NAME = "meta/llama-3.1-8b-instruct"
INPUT_ARTIFACT_NAME = "curated-sft-dataset:latest"

print("✅ Libraries and configurations loaded successfully.")

✅ Libraries and configurations loaded successfully.


### 1. Execution of Structured LLM-Judge Evaluation with End-to-End W&B Lineage

In [2]:
def evaluate_with_llm_judge():
    # Initialize Weights & Biases Run with explicit job_type for End-to-End Lineage
    run = wandb.init(
        project="nemo-fraud-detection",
        job_type="llm-judge-evaluation",
        name="llm-judge-evaluation",
        tags=["stage-3-llm-judge", "nim", "evaluation"],
        config={
            "eval_pipeline": "llm-as-a-judge",
            "model_name": MODEL_NAME,
            "input_artifact": INPUT_ARTIFACT_NAME
        }
    )
    
    # Consume upstream artifact from Curation step to establish lineage link
    print(f"📥 Consuming artifact '{INPUT_ARTIFACT_NAME}' from Weights & Biases...")
    artifact = run.use_artifact(INPUT_ARTIFACT_NAME)
    artifact_dir = artifact.download()
    val_file_path = Path(artifact_dir) / "validation.jsonl"
    
    print(f"🚀 Starting evaluation with structured Pandas analysis & W&B tracking on {val_file_path.name}...\n")
    
    results_list = []
    sample_log_lines = []
    total = 0
    passed = 0
    
    with open(val_file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            transcript_text = data.get("input", "")
            true_label = data.get("output", "").strip().lower()
            
            if not transcript_text:
                continue
            
            total += 1
            
            # Step 1: The Working Model (Worker)
            worker_payload = {
                "model": MODEL_NAME,
                "messages": [
                    {
                        "role": "system",
                        "content": (
                            "You are an expert in fraud detection for customer support calls. "
                            "Analyze the transcript strictly according to the following format:\n"
                            "1. Emotions/Pressure: [Description]\n"
                            "2. Security Risk: [Description]\n"
                            "Result: [fraud or legitimate]"
                        )
                    },
                    {
                        "role": "user",
                        "content": "Customer: Hello, I just wanted to ask when my next invoice will be debited. Agent: Good day, the debit always takes place at the end of the month."
                    },
                    {
                        "role": "assistant",
                        "content": "1. Emotions/Pressure: Calm, normal, no pressure.\n2. Security Risk: None, simple billing inquiry.\nResult: legitimate"
                    },
                    {
                        "role": "user",
                        "content": "Customer: My account is completely emptied out, release the TAN immediately!"
                    },
                    {
                        "role": "assistant",
                        "content": "1. Emotions/Pressure: Extremely panicked, heavy pressure on the agent for immediate release.\n2. Security Risk: Bypassing security standards, suspicion of fraud.\nResult: fraud"
                    },
                    {
                        "role": "user",
                        "content": transcript_text
                    }
                ],
                "temperature": 0.0,
                "max_tokens": 150
            }
            
            try:
                worker_response = requests.post(NIM_URL, json=worker_payload)
                worker_result = worker_response.json()
                model_output = worker_result["choices"][0]["message"]["content"].strip()
                
                # Extracting the result from the model output
                if "result: legitimate" in model_output.lower() and "fraud" not in model_output.lower():
                    model_answer = "legitimate"
                elif "result: fraud" in model_output.lower() and "legitimate" not in model_output.lower():
                    model_answer = "fraud"
                else:
                    match = re.search(r'\b(fraud|legitimate)\b', model_output.lower())
                    model_answer = match.group(1) if match else "unknown"
                
                # Step 2: The LLM Judge evaluates the answer
                judge_payload = {
                    "model": MODEL_NAME,
                    "messages": [
                        {
                            "role": "system",
                            "content": (
                                "You are a strict quality inspector for AI-assisted fraud detection. "
                                "Check whether the AI model's analysis arrives at the same 'Result' as the expected ground truth label.\n"
                                "Reply exclusively in this format:\n"
                                "Score: PASS (if correct) or Score: FAIL (if incorrect)\n"
                                "Reasoning: [Short text]"
                            )
                        },
                        {
                            "role": "user",
                            "content": (
                                f"Transcript: {transcript_text}\n"
                                f"Expected Label (Ground Truth): {true_label}\n"
                                f"AI Model Response:\n{model_output}"
                            )
                        }
                    ],
                    "temperature": 0.0,
                    "max_tokens": 100
                }
                
                judge_response = requests.post(NIM_URL, json=judge_payload)
                judge_result = judge_response.json()
                judge_output = judge_result["choices"][0]["message"]["content"].strip()
                
                is_pass = "score: pass" in judge_output.lower()
                if is_pass:
                    passed += 1
                
                # Store row for the DataFrame
                results_list.append({
                    "id": total,
                    "true_label": true_label,
                    "model_prediction": model_answer,
                    "judge_score": "PASS" if is_pass else "FAIL",
                    "model_output": model_output.replace("\n", " ")
                })
                
                log_line = f"Entry {total:02d} | Judge: {'✅ PASS' if is_pass else '❌ FAIL'}"
                sample_log_lines.append(log_line)
                print(log_line)
                
            except Exception as e:
                print(f"Error processing entry {total}: {e}")

    # Create DataFrame from results
    df = pd.DataFrame(results_list)
    
    success_rate = (passed / total) * 100 if total > 0 else 0
    print("\n" + "="*50)
    print(f"✅ Successfully Evaluated (PASS): {passed} / {total}")
    print(f"🎯 Final Judge Success Rate: {success_rate:.2f}%")
    print("="*50)

    # Ensure directory and save report as CSV
    os.makedirs("/data/nemo-fraud-detection-notebooks/notebooks/03_Evaluation/data", exist_ok=True)
    csv_path = "/data/nemo-fraud-detection-notebooks/notebooks/03_Evaluation/data/evaluation_report.csv"
    df.to_csv(csv_path, index=False)
    print(f"📁 Detailed report saved to: {csv_path}")

    # ==========================================
    # 🚀 Weights & Biases Logging & Artifacts
    # ==========================================
    run.log({
        "eval/total_samples": total,
        "eval/passed_samples": passed,
        "eval/failed_samples": total - passed,
        "eval/judge_success_rate_pct": success_rate
    })

    # Interactive W&B Table
    wandb_table = wandb.Table(dataframe=df)
    run.log({"eval/detailed_results_table": wandb_table})

    # Version CSV report as an Artifact with rich metadata
    artifact = wandb.Artifact(
        name="llm-judge-evaluation-report",
        type="evaluation-report",
        description="Detailed LLM-as-a-Judge evaluation results and reasoning for fraud detection",
        metadata={
            "stage": "03_evaluation",
            "total_samples": total,
            "passed_samples": passed,
            "success_rate_pct": round(success_rate, 2)
        }
    )
    artifact.add_file(csv_path)
    
    logged_artifact = run.log_artifact(artifact, aliases=["latest", "eval-v1"])
    logged_artifact.wait()
    
    run.finish()
    print("✅ W&B evaluation lineage registered successfully!")

    # Render output inside a collapsible HTML block labeled 'Sample Output'
    summary_text = (
        "=============================================\n"
        "📊 LLM-AS-A-JUDGE EVALUATION RESULTS\n"
        "=============================================\n"
        f"✅ Successfully Evaluated (PASS): {passed} / {total}\n"
        f"🎯 Final Judge Success Rate: {success_rate:.2f}%\n"
        "=============================================\n\n"
        "Detailed Samples:\n" +
        "\n".join(sample_log_lines)
    )
    
    escaped_output = html.escape(summary_text)
    html_widget = f"""
<details style="border: 1px solid #d0d7de; border-radius: 6px; padding: 10px 14px; background-color: #f6f8fa; margin-top: 10px;" open>
    <summary style="cursor: pointer; font-weight: bold; color: #0969da; font-size: 14px;">
        ▶ Sample Output
    </summary>
    <pre style="margin-top: 12px; white-space: pre-wrap; font-size: 13px; font-family: monospace; background: #ffffff; padding: 10px; border: 1px solid #e1e4e8; border-radius: 4px;">{escaped_output}</pre>
</details>
"""
    display(HTML(html_widget))

# Start LLM-Judge Evaluation
evaluate_with_llm_judge()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /root/.netrc.
wandb: Currently logged in as: lohmann-daniel (lohmann-daniel-myself) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


📥 Consuming artifact 'curated-sft-dataset:latest' from Weights & Biases...


wandb:   3 of 3 files downloaded.  


🚀 Starting evaluation with structured Pandas analysis & W&B tracking on validation.jsonl...

Entry 01 | Judge: ❌ FAIL
Entry 02 | Judge: ✅ PASS
Entry 03 | Judge: ✅ PASS
Entry 04 | Judge: ✅ PASS
Entry 05 | Judge: ✅ PASS
Entry 06 | Judge: ✅ PASS
Entry 07 | Judge: ❌ FAIL
Entry 08 | Judge: ✅ PASS
Entry 09 | Judge: ✅ PASS
Entry 10 | Judge: ✅ PASS
Entry 11 | Judge: ✅ PASS
Entry 12 | Judge: ✅ PASS

✅ Successfully Evaluated (PASS): 10 / 12
🎯 Final Judge Success Rate: 83.33%
📁 Detailed report saved to: /data/nemo-fraud-detection-notebooks/notebooks/03_Evaluation/data/evaluation_report.csv


eval/failed_samples,▁
eval/judge_success_rate_pct,▁
eval/passed_samples,▁
eval/total_samples,▁
eval/failed_samples,2
eval/judge_success_rate_pct,83.33333
eval/passed_samples,10
eval/total_samples,12


✅ W&B evaluation lineage registered successfully!
